# ML Framework: Hugging Face Sentiment Monitor
**Name:** Arantxa Sane 
**PRN:** 1302250199  
**Class:** S Y BCom FinTech | AI ML 1 - CCA2

In [ ]:
!pip -q install transformers pandas openpyxl

## Original examples

In [ ]:
from transformers import pipeline
sentiment = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

result = sentiment("I really enjoyed the picnic.")
print(result)

result = sentiment("The laboratory computers are very slow.")
print(result)

result = sentiment("Is the AI always correct?")
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.999864935874939}]
[{'label': 'NEGATIVE', 'score': 0.9997736811637878}]
[{'label': 'NEGATIVE', 'score': 0.5890882015228271}]


## Problem Statement 1: Banking app review monitor
A bank wants to know how customers feel about its mobile app. Classify each review and print the label with its confidence.

In [ ]:
reviews = [
    "The UPI payment went through instantly, great app!",
    "The app crashes every time I try to check my balance.",
    "Customer support solved my card issue within minutes.",
    "Too many hidden charges, I am very disappointed.",
]
for text, r in zip(reviews, sentiment(reviews)):
    print(f"{r['label']:8} {r['score']:.2f} | {text}")

POSITIVE 1.00 | The UPI payment went through instantly, great app!
NEGATIVE 1.00 | The app crashes every time I try to check my balance.
NEGATIVE 0.99 | Customer support solved my card issue within minutes.
NEGATIVE 1.00 | Too many hidden charges, I am very disappointed.


## Problem Statement 2: Financial news headline sentiment
An investor wants a quick read on market headlines. Score each headline and compute an overall mood (share of positive headlines).

In [ ]:
headlines = [
    "Sensex surges 800 points as foreign investors return",
    "RBI keeps repo rate unchanged, markets react calmly",
    "Banking stocks tumble after surprise rise in bad loans",
    "Fintech startup raises record funding round",
    "Inflation hits six-month high, consumers worry",
]
results = sentiment(headlines)
for h, r in zip(headlines, results):
    print(f"{r['label']:8} {r['score']:.2f} | {h}")

positive = sum(r["label"] == "POSITIVE" for r in results)
print(f"\nOverall mood: {positive}/{len(results)} headlines positive ({positive/len(results):.0%})")

POSITIVE 1.00 | Sensex surges 800 points as foreign investors return
NEGATIVE 0.92 | RBI keeps repo rate unchanged, markets react calmly
NEGATIVE 1.00 | Banking stocks tumble after surprise rise in bad loans
POSITIVE 1.00 | Fintech startup raises record funding round
NEGATIVE 1.00 | Inflation hits six-month high, consumers worry

Overall mood: 2/5 headlines positive (40%)


## Problem Statement 3: Complaint triage with a confidence threshold
A support team should escalate strongly negative complaints, and send low-confidence ones to a human (the model is not always correct).

In [ ]:
complaints = [
    "My money was deducted twice and nobody is replying!",
    "The statement download is a bit slow.",
    "Is the new interest rate applicable to old accounts?",
    "Excellent service, my loan was approved quickly.",
]
THRESHOLD = 0.90
for text, r in zip(complaints, sentiment(complaints)):
    if r["score"] < THRESHOLD:
        action = "SEND TO HUMAN REVIEW (low confidence)"
    elif r["label"] == "NEGATIVE":
        action = "ESCALATE"
    else:
        action = "NO ACTION"
    print(f"{action:38} | {r['label']} {r['score']:.2f} | {text}")

ESCALATE                               | NEGATIVE 1.00 | My money was deducted twice and nobody is replying!
ESCALATE                               | NEGATIVE 1.00 | The statement download is a bit slow.
ESCALATE                               | NEGATIVE 1.00 | Is the new interest rate applicable to old accounts?
NO ACTION                              | POSITIVE 1.00 | Excellent service, my loan was approved quickly.


## Problem Statement 4: Batch sentiment from an Excel file
Read customer feedback from an `.xlsx` file, add Sentiment and Confidence columns, and save the result to a new Excel file. (The input file is created below so the notebook is self-contained; upload your own `feedback_input.xlsx` to use different data.)

In [ ]:
import pandas as pd

sample = pd.DataFrame({
    "CustomerID": [101, 102, 103, 104, 105],
    "Feedback": [
        "Very smooth KYC process, loved it.",
        "Loan approval took far too long.",
        "The new dashboard looks clean and useful.",
        "I was charged a fee I was never told about.",
        "Support team was polite and helpful.",
    ],
})
sample.to_excel("feedback_input.xlsx", index=False)

df = pd.read_excel("feedback_input.xlsx")
out = sentiment(df["Feedback"].tolist())
df["Sentiment"] = [r["label"] for r in out]
df["Confidence"] = [round(r["score"], 3) for r in out]
df.to_excel("feedback_output.xlsx", index=False)
df

,CustomerID,Feedback,Sentiment,Confidence
0,101,"Very smooth KYC process, loved it.",POSITIVE,1.000
1,102,Loan approval took far too long.,NEGATIVE,0.999
2,103,The new dashboard looks clean and useful.,POSITIVE,0.998
3,104,I was charged a fee I was never told about.,NEGATIVE,0.933
4,105,Support team was polite and helpful.,POSITIVE,1.000


## Problem Statement 5: Sentiment summary report
Summarise the Excel results: count of each label and average confidence, as a manager would want to see.

In [ ]:
summary = df.groupby("Sentiment")["Confidence"].agg(["count", "mean"]).round(3)
print(summary)
print("\nMost negative feedback:")
print(df[df["Sentiment"] == "NEGATIVE"].sort_values("Confidence", ascending=False).head(3)[["CustomerID", "Feedback", "Confidence"]])

           count   mean
Sentiment              
NEGATIVE       2  0.966
POSITIVE       3  0.999

Most negative feedback:
   CustomerID                                     Feedback  Confidence
1         102             Loan approval took far too long.       0.999
3         104  I was charged a fee I was never told about.       0.933


## Problem Statement 6

A fintech company sends EMI and bill reminders to customers. Harsh-sounding drafts can upset customers and cause complaints.
Before sending, check each draft's tone and flag any that sound negative so the team can rewrite them.

In [ ]:
drafts = [
    "Thank you for banking with us! Your EMI of Rs 4,500 is due on the 5th.",
    "You have failed to pay your EMI. Pay immediately or face penalties.",
    "Friendly reminder: Your credit card bill is ready, and paying early saves you interest.",
    "Your payment is overdue and your account will be blocked.",
]

SAFE_SCORE = 0.80  # minimum confidence to trust a POSITIVE tone

for text, r in zip(drafts, sentiment(drafts)):
    if r["label"] == "POSITIVE" and r["score"] >= SAFE_SCORE:
        decision = "SEND"
    else:
        decision = "REWRITE (tone too harsh)"
    print(f"{decision:25} | {r['label']} {r['score']:.2f} | {text}")

SEND                      | POSITIVE 1.00 | Thank you for banking with us! Your EMI of Rs 4,500 is due on the 5th.
REWRITE (tone too harsh)  | NEGATIVE 1.00 | You have failed to pay your EMI. Pay immediately or face penalties.
SEND                      | POSITIVE 1.00 | Friendly reminder: Your credit card bill is ready, and paying early saves you interest.
REWRITE (tone too harsh)  | NEGATIVE 1.00 | Your payment is overdue and your account will be blocked.


## Problem Statement 7: Podcast episode audience reaction tracker
A podcast team wants to compare audience reactions to different episodes. Classify listener comments as positive or negative, then calculate the positive-comment percentage and average model confidence for each episode. This helps the team see which episodes received a warmer response.

In [ ]:
import pandas as pd

podcast_feedback = pd.DataFrame({
    "Episode": ["Episode 1", "Episode 1", "Episode 1", "Episode 2", "Episode 2", "Episode 2", "Episode 3", "Episode 3", "Episode 3"],
    "Comment": [
        "The guest shared such useful advice.",
        "The conversation felt natural and entertaining.",
        "The audio was distracting and the episode dragged.",
        "I learned a lot from this discussion.",
        "The topic was confusing and poorly explained.",
        "Loved the examples and the host's questions.",
        "This was the best episode so far.",
        "The pacing was slow and I lost interest.",
        "The ending was thoughtful and memorable."
    ]
})

predictions = sentiment(podcast_feedback["Comment"].tolist())
podcast_feedback["Sentiment"] = [p["label"] for p in predictions]
podcast_feedback["Confidence"] = [round(p["score"], 3) for p in predictions]
podcast_feedback["Positive_Flag"] = podcast_feedback["Sentiment"] == "POSITIVE"

episode_report = podcast_feedback.groupby("Episode").agg(
    Total_Comments=("Comment", "count"),
    Positive_Comments=("Positive_Flag", "sum"),
    Positive_Rate=("Positive_Flag", "mean"),
    Average_Confidence=("Confidence", "mean")
)
episode_report["Positive_Rate"] = (episode_report["Positive_Rate"] * 100).round(1)
episode_report["Average_Confidence"] = episode_report["Average_Confidence"].round(3)

print("Episode-level audience reaction:")
print(episode_report.sort_values("Positive_Rate", ascending=False))
print("\nIndividual predictions:")
print(podcast_feedback[["Episode", "Sentiment", "Confidence", "Comment"]].to_string(index=False))

## Problem Statement 8: Electric vehicle charging station experience monitor
An EV charging network wants to identify charging locations that may need attention. Classify driver comments and calculate the negative-feedback rate for each station. Flag stations with a high share of negative comments for a manual operational review; the sentiment result is only a screening signal, not proof of a station fault.

In [ ]:
import pandas as pd

ev_feedback = pd.DataFrame({
    "Station": ["Central Mall", "Central Mall", "Central Mall", "Riverside Hub", "Riverside Hub", "Riverside Hub", "Airport Road", "Airport Road", "Airport Road"],
    "Feedback": [
        "The charger worked perfectly and was easy to use.",
        "Fast charging and clear instructions, very convenient.",
        "The connector was faulty and charging stopped twice.",
        "The station was clean and the charging process was simple.",
        "I had to wait a long time for an available charger.",
        "The screen was broken and payment would not go through.",
        "Great location and the charger started immediately.",
        "The charging speed was disappointing.",
        "The station was out of service when I arrived."
    ]
})

predictions = sentiment(ev_feedback["Feedback"].tolist())
ev_feedback["Sentiment"] = [p["label"] for p in predictions]
ev_feedback["Confidence"] = [round(p["score"], 3) for p in predictions]
ev_feedback["Negative_Flag"] = ev_feedback["Sentiment"] == "NEGATIVE"

station_report = ev_feedback.groupby("Station").agg(
    Total_Comments=("Feedback", "count"),
    Negative_Comments=("Negative_Flag", "sum"),
    Negative_Rate=("Negative_Flag", "mean")
)
station_report["Negative_Rate"] = (station_report["Negative_Rate"] * 100).round(1)
station_report["Review_Flag"] = station_report["Negative_Rate"].apply(
    lambda rate: "REVIEW STATION" if rate >= 50 else "No automatic flag"
)

print("Charging station feedback summary:")
print(station_report.sort_values("Negative_Rate", ascending=False))
print("\nComment-level predictions:")
print(ev_feedback[["Station", "Sentiment", "Confidence", "Feedback"]].to_string(index=False))

## Problem Statement 9: Museum accessibility feedback analyzer
A museum wants to understand visitors' reactions to accessibility facilities such as ramps, lifts, captions, seating, and signage. Classify each comment and use simple keyword rules to assign a likely facility category. Summarise positive and negative comments by category so staff can decide where to investigate further. The category is assigned by keywords, not by the sentiment model.

In [ ]:
import pandas as pd

museum_feedback = pd.DataFrame({
    "Feedback": [
        "The ramp near the entrance made access easy.",
        "The lift was out of order and we could not reach the upper floor.",
        "The captions on the video displays were clear and helpful.",
        "There were not enough benches to rest between galleries.",
        "The signs were easy to follow throughout the building.",
        "The accessible toilet was difficult to locate.",
        "The lift worked well and moved smoothly.",
        "The captions were too small to read from a distance."
    ]
})

facility_keywords = {
    "Ramp/access route": ["ramp", "entrance", "access"],
    "Lift": ["lift", "upper floor"],
    "Captions": ["caption", "video displays", "read"],
    "Seating": ["bench", "benches", "rest", "seat"],
    "Signage": ["sign", "signs", "locate", "toilet"]
}

def detect_facility(text):
    text_lower = text.lower()
    matches = [category for category, keywords in facility_keywords.items()
               if any(word in text_lower for word in keywords)]
    return ", ".join(matches) if matches else "Other"

predictions = sentiment(museum_feedback["Feedback"].tolist())
museum_feedback["Sentiment"] = [p["label"] for p in predictions]
museum_feedback["Confidence"] = [round(p["score"], 3) for p in predictions]
museum_feedback["Facility_Category"] = museum_feedback["Feedback"].apply(detect_facility)

facility_report = museum_feedback.groupby("Facility_Category").agg(
    Total_Comments=("Feedback", "count"),
    Positive_Comments=("Sentiment", lambda s: (s == "POSITIVE").sum()),
    Negative_Comments=("Sentiment", lambda s: (s == "NEGATIVE").sum())
)

print("Accessibility feedback by facility:")
print(facility_report.sort_values("Negative_Comments", ascending=False))
print("\nIndividual comments:")
print(museum_feedback[["Facility_Category", "Sentiment", "Confidence", "Feedback"]].to_string(index=False))

## Problem Statement 10: Mobile game beta-version sentiment comparison
A game studio has released two beta builds of a puzzle game and wants to compare tester reactions before deciding what to improve. Classify feedback for each build and compare the positive-feedback rate and average model confidence. This is a quick comparison of written reactions, not a substitute for bug reports or gameplay testing.

In [ ]:
import pandas as pd

beta_feedback = pd.DataFrame({
    "Build": ["Build A", "Build A", "Build A", "Build A", "Build B", "Build B", "Build B", "Build B"],
    "Tester_ID": ["T01", "T02", "T03", "T04", "T05", "T06", "T07", "T08"],
    "Feedback": [
        "The puzzles are clever and satisfying to solve.",
        "The controls feel clunky and sometimes do not respond.",
        "I like the art style and the levels are fun.",
        "The game froze twice during the tutorial.",
        "The new controls feel smooth and responsive.",
        "The updated levels are more creative and enjoyable.",
        "The game still crashes when I open the map.",
        "The menu is confusing and hard to navigate."
    ]
})

predictions = sentiment(beta_feedback["Feedback"].tolist())
beta_feedback["Sentiment"] = [p["label"] for p in predictions]
beta_feedback["Confidence"] = [round(p["score"], 3) for p in predictions]
beta_feedback["Positive_Flag"] = beta_feedback["Sentiment"] == "POSITIVE"

build_report = beta_feedback.groupby("Build").agg(
    Tester_Comments=("Feedback", "count"),
    Positive_Comments=("Positive_Flag", "sum"),
    Positive_Rate=("Positive_Flag", "mean"),
    Average_Confidence=("Confidence", "mean")
)
build_report["Positive_Rate"] = (build_report["Positive_Rate"] * 100).round(1)
build_report["Average_Confidence"] = build_report["Average_Confidence"].round(3)

print("Beta build comparison:")
print(build_report)
if build_report["Positive_Rate"].nunique() > 1:
    preferred_build = build_report["Positive_Rate"].idxmax()
    print(f"\nHigher positive-feedback rate: {preferred_build}")
else:
    print("\nBoth builds have the same positive-feedback rate.")

print("\nTester-level results:")
print(beta_feedback[["Build", "Tester_ID", "Sentiment", "Confidence", "Feedback"]].to_string(index=False))